# Stage 6 — Circuit Analysis

**Two experiments using only frozen CTLS checkpoint weights — no new training.**

1. **Activation Maximization ("Circuit Dreams")** — optimise input pixels from random noise to minimise cosine distance to a target point in circuit space. Reveals what visual patterns the model's circuits actually encode.

2. **Circuit Neighbors vs Output Neighbors** — for boundary images, compare the 5-nearest neighbours in circuit space against the 5-nearest neighbours in output (softmax) space. The divergence cases show where circuit structure predicts something different from the classification output.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from evaluation.circuit_analysis import CircuitAnalyzer, denormalize, CIFAR10_CLASSES
from data.cifar import get_standard_loaders

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

with open('configs/ctls.yaml') as f:
    config = yaml.safe_load(f)

mcfg = config['model']
ecfg = mcfg['meta_encoder']

# Use the final annealed temperature (0.1), not the init value in ctls.yaml.
# SoftMask.temperature is not saved as a checkpoint buffer.
soft_mask = SoftMask(init_temperature=0.1)

backbone = CTLSBackbone(
    arch=mcfg['arch'],
    num_classes=mcfg['num_classes'],
    soft_mask=soft_mask,
    pretrained=False,
).to(DEVICE)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    hidden_dim=ecfg.get('hidden_dim', 256),
    embedding_dim=ecfg.get('embedding_dim', 64),
    encoder_type=ecfg.get('encoder_type', 'mlp'),
).to(DEVICE)

ckpt = torch.load('experiments/ctls/best.pt', map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state'])
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
backbone.eval()
meta_encoder.eval()

for p in backbone.parameters():     p.requires_grad_(False)
for p in meta_encoder.parameters(): p.requires_grad_(False)

print(f"Loaded checkpoint: epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=128,
    num_workers=dcfg.get('num_workers', 4),
    augment=False,
    download=True,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, DEVICE)
print("Analyzer ready.")

In [ ]:
print("Collecting circuit embeddings and output probabilities...")
z_all, logits_all, x_all, labels_all = analyzer.collect_all(max_samples=10000)

print(f"  {len(z_all)} samples — circuit embedding: {tuple(z_all.shape)}")

centroids = analyzer.class_centroids(z_all, labels_all)

# Sanity check: nearest-centroid accuracy
nearest_cls = torch.stack(
    [z_all @ centroids[c] for c in range(10)], dim=1
).argmax(dim=1)
centroid_acc = (nearest_cls == labels_all).float().mean()
print(f"  Nearest-centroid accuracy: {centroid_acc:.4f}")

---

## 1. Activation Maximization ("Circuit Dreams")

Starting from small random noise, we run gradient descent through the *frozen* backbone and meta-encoder to minimise cosine distance between the generated image's circuit embedding and a target point in circuit space:

```
x_opt = argmin  1 - cos_sim(meta_encoder(backbone(x)), target_z)
         x      + λ_tv · TV(x) + λ_l2 · ||x||²
```

TV and L₂ regularisation suppress adversarial high-frequency noise.

**Three target types:**
- **Class centroid** — what does the model's circuit think each class looks like?
- **Interpolation step** — what is the model's internal ambiguity zone between two classes?
- **Confusion-zone point** — what does the circuit encode in a real image it handles ambiguously?

### 1.1 Class Centroid Dreams

500 optimisation steps per class. The resulting images reveal what colour palette, texture, and (if the circuit encodes it) spatial structure each class centroid corresponds to.

In [ ]:
os.makedirs('experiments/stage6', exist_ok=True)

print("Generating class centroid dreams (512 steps each)...")
dream_centroids = {}
for cls in range(10):
    print(f"  [{cls+1:2d}/10] {CIFAR10_CLASSES[cls]}...", end=' ', flush=True)
    dream_centroids[cls] = analyzer.activate_maximize(centroids[cls])
    print("done")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Stage 6 — Circuit Dreams: Class Centroids', fontsize=13)
for cls in range(10):
    ax = axes[cls // 5, cls % 5]
    ax.imshow(dream_centroids[cls].permute(1, 2, 0).numpy())
    ax.set_title(CIFAR10_CLASSES[cls], fontsize=10)
    ax.axis('off')
fig.tight_layout()
fig.savefig('experiments/stage6/dreams_centroids.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.2 Interpolation Paths

Linearly interpolate between two class centroids in circuit space (L2-normalised at each step), then run activation maximization for each step. The resulting images reveal the shared visual structure that makes the model treat two classes as related.

Pairs: **dog ↔ cat**, **automobile ↔ truck**, **ship ↔ airplane**.

In [ ]:
pairs = [
    (5, 3, 'dog_cat',          'dog',        'cat'),
    (1, 9, 'automobile_truck', 'automobile', 'truck'),
    (8, 0, 'ship_airplane',    'ship',       'airplane'),
]
N_STEPS = 8

for cls_a, cls_b, fname, name_a, name_b in pairs:
    print(f"Interpolating {name_a} \u2194 {name_b}...")
    z_a = centroids[cls_a].to(DEVICE)
    z_b = centroids[cls_b].to(DEVICE)
    alphas = torch.linspace(0, 1, N_STEPS, device=DEVICE)

    interp_imgs = []
    for i, alpha in enumerate(alphas):
        target = F.normalize((1 - alpha) * z_a + alpha * z_b, dim=-1)
        interp_imgs.append(analyzer.activate_maximize(target, n_steps=300))
        print(f"  step {i+1}/{N_STEPS}", end='\r')
    print()

    fig, axes = plt.subplots(1, N_STEPS, figsize=(N_STEPS * 1.7, 2.3))
    fig.suptitle(f'Circuit Dream Interpolation: {name_a} \u2192 {name_b}', fontsize=11)
    for i, (img, alpha) in enumerate(zip(interp_imgs, alphas)):
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].axis('off')
        if i == 0:
            axes[i].set_title(name_a, fontsize=9)
        elif i == N_STEPS - 1:
            axes[i].set_title(name_b, fontsize=9)
        else:
            axes[i].set_title(f'\u03b1={alpha.item():.2f}', fontsize=8)
    fig.tight_layout()
    fig.savefig(f'experiments/stage6/dreams_interp_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

### 1.3 Confusion Zone Dreams

For each class pair, find the real images from class A whose circuit embedding is closest (cosine distance) to class B's centroid — the most internally confused examples. Use each image's actual circuit embedding (not the centroid) as the optimisation target.

The resulting dream shows exactly what visual information drives the circuit-level confusion on that specific example.

**Top row:** original confused image. **Bottom row:** circuit dream of its embedding.

In [ ]:
confusion_pairs = [
    (3, 5, 'cat_dog',          'cat',        'dog'),
    (1, 9, 'automobile_truck', 'automobile', 'truck'),
    (4, 7, 'deer_horse',       'deer',       'horse'),
]
N_CONFUSION = 6

for cls_a, cls_b, fname, name_a, name_b in confusion_pairs:
    mask_a = labels_all == cls_a
    z_a    = z_all[mask_a]
    x_a    = x_all[mask_a]

    # Cosine distance to class B centroid — lower = more confused
    dists   = 1 - (z_a @ centroids[cls_b].cpu())
    closest = dists.argsort()[:N_CONFUSION]

    target_zs = z_a[closest]   # actual circuit embeddings of confused images
    orig_imgs  = x_a[closest]

    print(f"Confusion zone: {name_a} \u2192 {name_b}")
    dream_imgs = []
    for i in range(N_CONFUSION):
        dream_imgs.append(analyzer.activate_maximize(target_zs[i], n_steps=400))
        print(f"  image {i+1}/{N_CONFUSION}", end='\r')
    print()

    fig, axes = plt.subplots(2, N_CONFUSION, figsize=(N_CONFUSION * 1.7, 4.0))
    fig.suptitle(
        f'Confusion Zone: {name_a} images whose circuits most resemble {name_b}',
        fontsize=11,
    )
    for col in range(N_CONFUSION):
        axes[0, col].imshow(denormalize(orig_imgs[col]).permute(1, 2, 0).numpy())
        axes[0, col].axis('off')
        axes[1, col].imshow(dream_imgs[col].permute(1, 2, 0).numpy())
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel('Original', fontsize=9)
    axes[1, 0].set_ylabel('Circuit Dream', fontsize=9)
    fig.tight_layout()
    fig.savefig(f'experiments/stage6/dreams_confusion_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

---

## 2. Circuit Neighbors vs Output Neighbors

For each query image, retrieve:
- **5 circuit-space neighbors** — images with the most similar circuit embeddings (cosine distance on z)
- **5 output-space neighbors** — images with the most similar softmax distributions (cosine distance on class probabilities)

**When they agree:** the model's internal circuits and its final prediction are aligned on this image.

**When they diverge:** the model's circuits process this image like one class while the output layer votes for another — or vice versa. These cases reveal hidden circuit behaviour not visible in the classification output.

**Query selection:** 2 images per class with the highest *confusion score* — images whose circuit embedding is closer to a rival class centroid than to their own class centroid.

In [ ]:
K = 5  # neighbours per side

# Select 2 most circuit-confused images per class (20 total)
query_indices = []
for cls in range(10):
    mask       = labels_all == cls
    global_idx = torch.where(mask)[0]  # absolute indices into z_all
    z_cls      = z_all[mask]

    own_dist   = 1 - (z_cls @ centroids[cls].cpu())  # [M]
    other_dist = torch.stack(
        [1 - (z_cls @ centroids[c].cpu()) for c in range(10) if c != cls],
        dim=1,
    ).min(dim=1).values  # [M] — distance to nearest rival centroid

    # Positive confusion_score → this image is closer to a rival centroid than its own
    confusion_score = own_dist - other_dist
    top2 = confusion_score.argsort(descending=True)[:2]
    query_indices.extend(global_idx[top2].tolist())

query_indices = torch.tensor(query_indices)
print(f"Selected {len(query_indices)} query images (2 per class)")

# Pre-compute nearest centroid class for each query (for the summary table)
all_centroid_sims = torch.stack(
    [z_all @ centroids[c].cpu() for c in range(10)], dim=1
)
nearest_centroid_cls = all_centroid_sims.argmax(dim=1)

In [ ]:
def plot_knn_grid(q_indices, z_all, logits_all, x_all, labels_all, analyzer, k=5):
    """
    Grid: query image | k circuit-NN | k output-NN.
    A thin grey column separates the two NN groups.
    """
    n_q   = len(q_indices)
    n_col = 1 + k + k
    fig, axes = plt.subplots(n_q, n_col, figsize=(n_col * 1.35, n_q * 1.35))

    for row, qi in enumerate(q_indices.tolist()):
        q_z      = z_all[qi]
        q_logits = logits_all[qi]
        q_label  = labels_all[qi].item()

        _, c_imgs, _ = analyzer.knn_circuit(q_z, z_all, x_all, k=k)
        _, o_imgs, _ = analyzer.knn_output(q_logits, logits_all, x_all, k=k)

        # Query
        ax = axes[row, 0]
        ax.imshow(denormalize(x_all[qi]).permute(1, 2, 0).numpy())
        ax.axis('off')
        ax.set_ylabel(CIFAR10_CLASSES[q_label], fontsize=7, rotation=0,
                      labelpad=32, va='center')
        if row == 0:
            ax.set_title('Query', fontsize=8, fontweight='bold')

        # Circuit neighbours
        for j in range(k):
            ax = axes[row, 1 + j]
            ax.imshow(denormalize(c_imgs[j]).permute(1, 2, 0).numpy())
            ax.axis('off')
            if row == 0:
                ax.set_title(f'Circ-{j+1}', fontsize=7)

        # Output neighbours
        for j in range(k):
            ax = axes[row, 1 + k + j]
            ax.imshow(denormalize(o_imgs[j]).permute(1, 2, 0).numpy())
            ax.axis('off')
            if row == 0:
                ax.set_title(f'Out-{j+1}', fontsize=7)

    fig.subplots_adjust(wspace=0.04, hspace=0.04)
    return fig


print("Computing nearest neighbours...")
fig = plot_knn_grid(
    query_indices, z_all, logits_all, x_all, labels_all, analyzer, k=K
)
fig.suptitle(
    'Circuit Neighbors (Circ-1…5) vs Output Neighbors (Out-1…5)',
    fontsize=11, y=1.005,
)
fig.savefig('experiments/stage6/knn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Divergence summary: does the dominant class in circuit-NN match that in output-NN?
# "Dominant class" = the most common class among the k neighbours.
# This avoids false positives when both spaces agree on one class (returning
# only 1 unique class still counts as agreement, not divergence).

def _dominant_class(nn_idx):
    counts = torch.bincount(labels_all[nn_idx], minlength=10)
    return counts.argmax().item()

print(f"{'Idx':<6} {'True cls':<13} {'Near centroid':<14} "
      f"{'Circuit-NN classes':<30} {'Output-NN classes':<30} {'Dom C':<7} {'Dom O':<7} {'Status'}")
print('-' * 110)

for qi in query_indices.tolist():
    q_label  = labels_all[qi].item()
    q_z      = z_all[qi]
    q_logits = logits_all[qi]

    c_idx, _, _ = analyzer.knn_circuit(q_z, z_all, x_all, k=K)
    o_idx, _, _ = analyzer.knn_output(q_logits, logits_all, x_all, k=K)

    c_classes = set(labels_all[c_idx].tolist())
    o_classes = set(labels_all[o_idx].tolist())
    c_dom     = _dominant_class(c_idx)
    o_dom     = _dominant_class(o_idx)

    near_c = nearest_centroid_cls[qi].item()
    c_str  = ' '.join(CIFAR10_CLASSES[c][:4] for c in sorted(c_classes))
    o_str  = ' '.join(CIFAR10_CLASSES[c][:4] for c in sorted(o_classes))
    status = 'DIVERGE' if c_dom != o_dom else 'agree'

    print(f"{qi:<6} {CIFAR10_CLASSES[q_label]:<13} {CIFAR10_CLASSES[near_c]:<14} "
          f"{c_str:<30} {o_str:<30} "
          f"{CIFAR10_CLASSES[c_dom][:4]:<7} {CIFAR10_CLASSES[o_dom][:4]:<7} {status}")

---

## Summary

| Experiment | What it shows |
|---|---|
| Centroid dreams | What colour palette, texture, and spatial structure each class's circuit centroid encodes. Blurry output = semantic colour/texture only; sharper = spatial structure also present. |
| Interpolation dreams | The shared visual features in the model's internal ambiguity zone between two classes. |
| Confusion zone dreams | Which specific visual features drive circuit-level confusion on real boundary examples — the diagnostic payoff of having interpretable circuit space. |
| Circuit vs output KNN | Where circuit structure and final prediction agree or diverge. Divergence (≤1 class overlap) reveals images the model handles consistently in output but inconsistently in circuits, or vice versa. |

**Divergence interpretation:**
- **Circuit-NN ≠ Output-NN** → the model's circuits process this image similarly to a different class's examples, but the output layer still votes correctly. The mid-level feature similarity does not propagate to the final prediction.
- **Circuit-NN = Output-NN** → the model is internally consistent; circuit structure and final prediction are aligned.

**Next directions:**
- Idea 3 (Robustness Auditing): apply input noise and re-run activation maximization to visualise how perturbations shift the circuit representation.
- Idea 4 (Divergence Analysis): find the specific layer in the trajectory where the circuit diverges from the expected path for the true class.